In [1]:
import sys


In [2]:
print(sys.executable)

/Users/agastyanath/Downloads/ns_jupy/bin/python3.14


In [3]:
import networkx

In [70]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from networkx.algorithms import bipartite, community
import os, random
 
random.seed(42)
np.random.seed(42)

In [71]:
DATA_PATH = "./web-of-life_2026-03-23_105349/M_PL_001.csv"
OUT_DIR   = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)
 
df = pd.read_csv(DATA_PATH, index_col=0)
df = df.fillna(0).astype(float)
 
plants      = list(df.index)
pollinators = list(df.columns)
n_plants    = len(plants)
n_poll      = len(pollinators)
 
print(f"Dataset: {n_plants} plants × {n_poll} pollinators  |  "
      f"{int((df>0).sum().sum())} interactions")
 

Dataset: 84 plants × 101 pollinators  |  361 interactions


In [72]:
def build_bipartite(matrix_df):
    """Return a weighted bipartite NetworkX graph."""
    G = nx.Graph()
    pl  = list(matrix_df.index)
    pol = list(matrix_df.columns)
    G.add_nodes_from(pl,  bipartite=0, kind="plant")
    G.add_nodes_from(pol, bipartite=1, kind="pollinator")
    for p in pl:
        for q in pol:
            w = matrix_df.loc[p, q]
            if w > 0:
                G.add_edge(p, q, weight=float(w))
    return G
 
G_orig = build_bipartite(df)
print(f"Graph: {G_orig.number_of_nodes()} nodes, {G_orig.number_of_edges()} edges")
 

Graph: 185 nodes, 361 edges


In [81]:
def bipartite_modularity(G, communities, plant_nodes):
    m = G.number_of_edges()
    Q = 0.0

    for community in communities:
        comm_nodes = list(community)

        for idx_i in range(len(comm_nodes)):
            for idx_j in range(idx_i + 1, len(comm_nodes)):
                i = comm_nodes[idx_i]
                j = comm_nodes[idx_j]
                # Only consider cross-partition pairs
                if (i in plant_nodes and j not in plant_nodes) or \
                   (j in plant_nodes and i not in plant_nodes):

                    Aij = 1 if G.has_edge(i, j) else 0
                    ki = G.degree(i)
                    kj = G.degree(j)

                    Q += (Aij - (ki * kj) / m)

    return Q / m
    
def compute_metrics(G, poll_nodes):
    """Compute four centrality measures for every node."""
    deg  = nx.degree_centrality(G)
    btw  = nx.betweenness_centrality(G, weight="weight", normalized=True)
    # eigenvector needs connected graph; use largest component
    G_proj = bipartite.weighted_projected_graph(G, poll_nodes)

    eig = nx.eigenvector_centrality(G_proj, weight="weight")
 
    clust = {}
    for n in G.nodes():
        nbrs = list(G.neighbors(n))
        if len(nbrs) < 2:
            clust[n] = 0.0
        else:
            # bipartite clustering (Robins–Alexander)
            try:
                clust[n] = nx.clustering(G, n)
            except Exception:
                clust[n] = 0.0
 
    rows = []
    for n in poll_nodes:
        if n in G:
            rows.append({
                "species": n,
                "degree_centrality":      deg.get(n, 0),
                "betweenness_centrality": btw.get(n, 0),
                "eigenvector_centrality": eig.get(n, 0),
                "clustering_coefficient": clust.get(n, 0),
                "degree":                 G.degree(n),
            })
    return pd.DataFrame(rows).set_index("species")
 
metrics_df = compute_metrics(G_orig, pollinators)
print("\nTop-5 pollinators by degree centrality:")
print(metrics_df["degree_centrality"].sort_values(ascending=False).head())
 
# --- Nestedness (NODF) ---
def nodf_score(mat):
    """Compute NODF nestedness (0-100)."""
    mat = (mat > 0).astype(float).values
    R, C = mat.shape
    n_pairs, n_fill = 0, 0.0
    for i in range(R-1):
        for j in range(i+1, R):
            di, dj = mat[i].sum(), mat[j].sum()
            if di == dj or di == 0 or dj == 0:
                continue
            n_pairs += 1
            shared = (mat[i] * mat[j]).sum()
            n_fill += shared / min(di, dj)
    row_nodf = (n_fill / n_pairs * 100) if n_pairs else 0
 
    n_pairs, n_fill = 0, 0.0
    for i in range(C-1):
        for j in range(i+1, C):
            di, dj = mat[:, i].sum(), mat[:, j].sum()
            if di == dj or di == 0 or dj == 0:
                continue
            n_pairs += 1
            shared = (mat[:, i] * mat[:, j]).sum()
            n_fill += shared / min(di, dj)
    col_nodf = (n_fill / n_pairs * 100) if n_pairs else 0
    return (row_nodf + col_nodf) / 2
 
nestedness = nodf_score(df)
print(f"\nNestedness (NODF): {nestedness:.2f}")
 
# --- Modularity ---
def compute_modularity(G):
    for n, d in list(G.nodes(data=True))[:10]:
        print(n, d)
    communities = community.louvain_communities(G)
    plants = {n for n, d in G.nodes(data=True) if d.get('kind') == 'plant'}
    print("Number of plant nodes:", len(plants))
    Q_bip = bipartite_modularity(G, communities, plants)
    #comms = list(nx.community.greedy_modularity_communities(G, weight="weight"))
    #Q = nx.community.modularity(G, comms, weight="weight")
    return Q_bip, communities
 
Q, communities = compute_modularity(G_orig)
print(f"Modularity (Q): {Q:.4f}  |  Communities: {len(communities)}")
 


Top-5 pollinators by degree centrality:
species
Scaeva melanostoma          0.184783
Hymenoptera sp1 M_PL_001    0.119565
Diptera sp2 M_PL_001        0.114130
Megachile semirufa          0.081522
Megachile sp1 M_PL_001      0.070652
Name: degree_centrality, dtype: float64

Nestedness (NODF): 18.66
Acaena pinn {'bipartite': 0, 'kind': 'plant'}
Adesmia aconcaguensis {'bipartite': 0, 'kind': 'plant'}
Adesmia brachy {'bipartite': 0, 'kind': 'plant'}
Adesmia conferta {'bipartite': 0, 'kind': 'plant'}
Adesmia exilis {'bipartite': 0, 'kind': 'plant'}
Adesmia montana {'bipartite': 0, 'kind': 'plant'}
Adesmia radicifolia {'bipartite': 0, 'kind': 'plant'}
Alstroemeria pallida {'bipartite': 0, 'kind': 'plant'}
Anarthrophyllum cumingii {'bipartite': 0, 'kind': 'plant'}
Asteriscium aemocarpon {'bipartite': 0, 'kind': 'plant'}
Number of plant nodes: 84
Modularity (Q): 0.5067  |  Communities: 11


In [82]:
def classify_roles(metrics_df):
    """
    Classify each pollinator into one of four ecological roles:
    Generalist | Specialist | Connector | Keystone
    using k-means on the four centrality features.
    """
    feats = ["degree_centrality", "betweenness_centrality",
             "eigenvector_centrality", "clustering_coefficient"]
    X = metrics_df[feats].values
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
 
    km = KMeans(n_clusters=4, random_state=42, n_init=20)
    labels = km.fit_predict(Xs)
 
    centers = scaler.inverse_transform(km.cluster_centers_)
    # Assign role names based on cluster center characteristics
    role_map = {}
    for k in range(4):
        deg = centers[k, 0]
        btw = centers[k, 1]
        eig = centers[k, 2]
        if deg > np.median(centers[:, 0]) and eig > np.median(centers[:, 2]):
            role_map[k] = "Generalist"
        elif btw > np.median(centers[:, 1]) and deg < np.median(centers[:, 0]):
            role_map[k] = "Connector"
        elif eig > np.median(centers[:, 2]) and btw > np.median(centers[:, 1]):
            role_map[k] = "Keystone"
        else:
            role_map[k] = "Specialist"
 
    # Ensure unique roles
    assigned = list(role_map.values())
    all_roles = ["Generalist", "Specialist", "Connector", "Keystone"]
    used, final_map = set(), {}
    order_by_deg = sorted(range(4), key=lambda k: centers[k,0], reverse=True)
    priority = ["Generalist", "Keystone", "Connector", "Specialist"]
    for k, role in zip(order_by_deg, priority):
        final_map[k] = role
 
    metrics_df = metrics_df.copy()
    metrics_df["role"] = [final_map[l] for l in labels]
    return metrics_df
 
metrics_df = classify_roles(metrics_df)
print("\nSpecies roles:")
print(metrics_df["role"].value_counts())
 


Species roles:
role
Specialist    79
Connector     16
Keystone       5
Generalist     1
Name: count, dtype: int64


In [83]:
def cascading_extinction(df_mat, removal_order="degree", threshold=0.0):
    """
    Simulate sequential pollinator removal.
    A plant goes extinct when ALL its pollinators are removed.
    Returns lists of (fraction_pollinators_removed,
                      fraction_plants_surviving,
                      n_components).
    """
    mat    = df_mat.copy()
    poll_d = {p: int(mat[p].sum()) for p in mat.columns}
 
    if removal_order == "degree":
        order = sorted(poll_d, key=lambda x: poll_d[x], reverse=True)
    elif removal_order == "random":
        order = list(mat.columns)
        random.shuffle(order)
    else:
        order = list(mat.columns)
 
    n_poll_total  = len(order)
    n_plant_total = len(mat)
 
    frac_removed   = [0.0]
    frac_surviving = [1.0]
    n_components   = [1]
 
    G_sim = build_bipartite(mat)
 
    for i, poll in enumerate(order):
        mat.drop(columns=[poll], inplace=True)
        if poll in G_sim:
            G_sim.remove_node(poll)
 
        # secondary extinctions: plants with no remaining interactions
        dead_plants = [p for p in list(mat.index) if mat.loc[p].sum() == 0]
        for dp in dead_plants:
            mat.drop(index=[dp], inplace=True)
            if dp in G_sim:
                G_sim.remove_node(dp)
 
        frac_removed.append((i + 1) / n_poll_total)
        frac_surviving.append(len(mat) / n_plant_total)
        nc = nx.number_connected_components(G_sim) if G_sim.number_of_nodes() > 0 else 0
        n_components.append(nc)
 
    return frac_removed, frac_surviving, n_components
 
print("\nRunning cascading extinction simulations…")
fr_deg, fs_deg, nc_deg = cascading_extinction(df.copy(), removal_order="degree")
fr_rnd, fs_rnd, nc_rnd = cascading_extinction(df.copy(), removal_order="random")
 
 


Running cascading extinction simulations…


In [84]:
def climate_stress_simulation(df_mat, stress_levels=None):
    """
    Model environmental stress by multiplying interaction strengths
    by (1 - stress) and counting surviving interactions & connectivity.
    stress=0 → no change; stress=1 → total collapse.
    """
    if stress_levels is None:
        stress_levels = np.linspace(0, 1, 30)
 
    connectivities = []
    n_links        = []
    n_surv_poll    = []
    n_surv_plant   = []
 
    for s in stress_levels:
        mat_s = df_mat * (1 - s)
        # interactions below 10% of mean considered lost
        thr   = 0.1 * df_mat.values[df_mat.values > 0].mean()
        mat_s = mat_s.where(mat_s >= thr, 0)
 
        G_s = build_bipartite(mat_s)
        total_possible = len(df_mat.index) * len(df_mat.columns)
        conn = G_s.number_of_edges() / total_possible if total_possible else 0
        connectivities.append(conn)
        n_links.append(G_s.number_of_edges())
        n_surv_poll.append(sum(1 for p in df_mat.columns
                               if mat_s[p].sum() > 0))
        n_surv_plant.append(sum(1 for p in df_mat.index
                                if mat_s.loc[p].sum() > 0))
 
    return stress_levels, connectivities, n_links, n_surv_poll, n_surv_plant
 
print("Running climate stress simulation…")
stress_lvls, conn_list, nlinks, ns_poll, ns_plant = climate_stress_simulation(df)
 

Running climate stress simulation…


In [85]:
def adaptive_rewiring_simulation(df_mat, rewire_prob=0.6):
    """
    Remove pollinators one by one (degree-descending).
    After each removal, remaining species may form new interactions
    with probability `rewire_prob`.
    Returns plant survival curves for rewired vs non-rewired.
    """
    mat_rw  = df_mat.copy().astype(float)
    mat_nrw = df_mat.copy().astype(float)
 
    order = sorted(df_mat.columns,
                   key=lambda p: df_mat[p].sum(), reverse=True)
 
    n_poll_total  = len(order)
    n_plant_total = len(df_mat)
 
    surv_rw, surv_nrw = [1.0], [1.0]
    steps = [0.0]
 
    for i, poll in enumerate(order):
        # ── no rewiring ──
        mat_nrw = mat_nrw.drop(columns=[poll], errors="ignore")
        dead_nrw = [p for p in mat_nrw.index if mat_nrw.loc[p].sum() == 0]
        mat_nrw.drop(index=dead_nrw, inplace=True)
 
        # ── with rewiring ──
        mat_rw = mat_rw.drop(columns=[poll], errors="ignore")
 
        # Plants that lost ALL pollinators attempt to acquire a new one
        dead_candidates = [p for p in mat_rw.index if mat_rw.loc[p].sum() == 0]
        for p in dead_candidates:
            surviving_pollinators = [c for c in mat_rw.columns]
            if surviving_pollinators and random.random() < rewire_prob:
                new_poll = random.choice(surviving_pollinators)
                mat_rw.loc[p, new_poll] = 1.0   # new weak interaction
 
        dead_rw = [p for p in mat_rw.index if mat_rw.loc[p].sum() == 0]
        mat_rw.drop(index=dead_rw, inplace=True)
 
        steps.append((i + 1) / n_poll_total)
        surv_nrw.append(len(mat_nrw) / n_plant_total)
        surv_rw.append(len(mat_rw)  / n_plant_total)
 
    return steps, surv_rw, surv_nrw
 
print("Running adaptive rewiring simulation…")
steps_rw, surv_rw, surv_nrw = adaptive_rewiring_simulation(df, rewire_prob=0.6)
 

Running adaptive rewiring simulation…


In [86]:
def early_warning_signals(df_mat, n_steps=40):
    """
    Progressively reduce mutualistic strength.
    Track variance and lag-1 autocorrelation of the
    'community biomass proxy' (total interaction strength).
    """
    stress_vals  = np.linspace(0, 0.98, n_steps)
    variances    = []
    autocorrs    = []
    total_str    = []
 
    for s in stress_vals:
        mat_s = df_mat * (1 - s)
        # time-like fluctuation: add small Gaussian noise
        ts    = []
        for t in range(30):
            noise = np.random.normal(0, 0.05 * (1 + 3*s), mat_s.shape)
            mat_t = np.maximum(mat_s.values + noise, 0)
            ts.append(mat_t.sum())
 
        ts = np.array(ts)
        variances.append(np.var(ts))
        if len(ts) > 1:
            ac = np.corrcoef(ts[:-1], ts[1:])[0, 1]
            autocorrs.append(ac if not np.isnan(ac) else 0.0)
        else:
            autocorrs.append(0.0)
        total_str.append(ts.mean())
 
    return stress_vals, variances, autocorrs, total_str
 
print("Computing early warning signals…")
ew_stress, ew_var, ew_ac, ew_total = early_warning_signals(df)
 

Computing early warning signals…


In [87]:
def predict_extinction_risk(metrics_df, df_mat):
    """
    Predict extinction-risk score for each pollinator using:
    - Degree, betweenness, eigenvector centrality, clustering
    - Relative abundance
    Risk is inverse of robustness; we use RF regression with
    heuristic labels derived from network position.
    """
    feats = ["degree_centrality", "betweenness_centrality",
             "eigenvector_centrality", "clustering_coefficient"]
    X     = metrics_df[feats].copy()

    print("degree unique:", X["degree_centrality"].nunique())
    print("eigen unique:", X["eigenvector_centrality"].nunique())
    print(metrics_df["eigenvector_centrality"].head())
    print(metrics_df["eigenvector_centrality"].describe())
 
    # Heuristic risk label: high degree + high eigen → low risk
    total_visits = df_mat.sum()
    rel_abund    = (total_visits / total_visits.sum()).reindex(metrics_df.index).fillna(0)
    X["rel_abundance"] = rel_abund.values
 
    # Risk = 1 / (degree * eigen + ε)  (normalised)
    risk_raw = 1.0 / (X["degree_centrality"] * X["eigenvector_centrality"] + 1e-6)
    risk_raw = (risk_raw - risk_raw.min()) / (risk_raw.max() - risk_raw.min() + 1e-9)
    X["risk_label"] = risk_raw.values
    print("risk_raw unique:", np.unique(risk_raw))
 
    # Feature importance via Random Forest (self-supervised)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X[feats + ["rel_abundance"]])
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    # Binarize into high/low risk
    threshold = np.percentile(risk_raw, 50)
    y = (risk_raw > threshold).astype(int)
    #y = (risk_raw > risk_raw.median()).astype(int)
    print("Unique classes in y:", np.unique(y))
    rf.fit(Xs, y)
 
    risk_score = rf.predict_proba(Xs)[:, 1]  # prob of high risk
    result     = metrics_df.copy()
    result["extinction_risk"] = risk_score
    result["rel_abundance"]   = rel_abund.values
 
    feat_importance = pd.Series(
        rf.feature_importances_,
        index=feats + ["rel_abundance"]
    ).sort_values(ascending=False)
 
    return result, feat_importance
 
print("Predicting extinction risk…")
risk_df, feat_imp = predict_extinction_risk(metrics_df, df)
print("\nTop vulnerable pollinators:")
print(risk_df[["extinction_risk","degree","role"]]
      .sort_values("extinction_risk", ascending=False).head(7))
print("\nFeature importances:\n", feat_imp)
 

Predicting extinction risk…
degree unique: 15
eigen unique: 81
species
Alloscirtetica gayi            0.050860
Alloscirtetica rufitarsis      0.126973
Alloscirtetica sp1 M_PL_001    0.210391
Andinus venustus               0.029236
Anthidium adriani              0.060895
Name: eigenvector_centrality, dtype: float64
count    1.010000e+02
mean     6.583248e-02
std      7.498497e-02
min      2.011675e-21
25%      1.328981e-02
50%      3.109153e-02
75%      9.057254e-02
max      3.745647e-01
Name: eigenvector_centrality, dtype: float64
risk_raw unique: [0.00000000e+00 2.56021610e-05 2.84606225e-05 2.91602701e-05
 3.81697436e-05 4.01350349e-05 7.42123415e-05 8.27175317e-05
 1.00129135e-04 1.05670021e-04 1.36267342e-04 1.43099009e-04
 1.54282624e-04 1.55612745e-04 1.66023598e-04 1.84447353e-04
 2.07509147e-04 2.17666219e-04 2.46886987e-04 2.96591155e-04
 3.46654792e-04 4.68369064e-04 5.59653869e-04 6.34270269e-04
 7.25707905e-04 7.58097848e-04 7.67831050e-04 7.87820783e-04
 8.53609892e-04 9.2

In [88]:
def community_jaccard(df_mat):
    """
    Compute plant-pollinator Jaccard similarity between
    all pollinator pairs (shared plant partners / union).
    """
    bin_mat = (df_mat > 0).astype(int)
    n       = len(bin_mat.columns)
    sim     = np.zeros((n, n))
    cols    = list(bin_mat.columns)
    for i in range(n):
        for j in range(i, n):
            a = set(bin_mat.index[bin_mat.iloc[:, i] == 1])
            b = set(bin_mat.index[bin_mat.iloc[:, j] == 1])
            inter = len(a & b)
            union = len(a | b)
            s     = inter / union if union > 0 else 0
            sim[i, j] = sim[j, i] = s
    return pd.DataFrame(sim, index=cols, columns=cols)
 
sim_matrix = community_jaccard(df)

In [89]:
ROLE_COLORS = {
    "Generalist": "#2196F3",
    "Keystone":   "#F44336",
    "Connector":  "#FF9800",
    "Specialist": "#4CAF50",
}
 
fig = plt.figure(figsize=(22, 28))
fig.patch.set_facecolor("#0D1117")
gs  = gridspec.GridSpec(4, 3, figure=fig,
                        hspace=0.48, wspace=0.38,
                        left=0.06, right=0.97,
                        top=0.95, bottom=0.04)
 
TXT  = "white"
GRID = "#333333"
 
def style_ax(ax, title=""):
    ax.set_facecolor("#161B22")
    ax.tick_params(colors=TXT, labelsize=8)
    ax.xaxis.label.set_color(TXT)
    ax.yaxis.label.set_color(TXT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID)
    if title:
        ax.set_title(title, color=TXT, fontsize=10, fontweight="bold", pad=6)
 

In [90]:
ax1 = fig.add_subplot(gs[0, 0])
style_ax(ax1, "Bipartite Pollination Network")
 
plant_x  = [0.05] * n_plants
plant_y  = np.linspace(0.02, 0.98, n_plants)
poll_x   = [0.95] * n_poll
poll_y   = np.linspace(0.05, 0.95, n_poll)
pos      = {plants[i]: (plant_x[i], plant_y[i]) for i in range(n_plants)}
pos.update({pollinators[i]: (poll_x[i], poll_y[i]) for i in range(n_poll)})
 
poll_color = [ROLE_COLORS.get(metrics_df.loc[p, "role"], "#888")
              if p in metrics_df.index else "#888" for p in pollinators]
 
nx.draw_networkx_nodes(G_orig, pos, nodelist=plants,
                       node_color="#66BB6A", node_size=30,
                       alpha=0.8, ax=ax1)
nx.draw_networkx_nodes(G_orig, pos, nodelist=pollinators,
                       node_color=poll_color, node_size=60,
                       alpha=0.9, ax=ax1)
nx.draw_networkx_edges(G_orig, pos, alpha=0.15,
                       edge_color="#90CAF9", width=0.5, ax=ax1)
short_labels = {p: p[:10] for p in pollinators}
nx.draw_networkx_labels(G_orig, pos, labels=short_labels,
                        font_size=4.5, font_color=TXT, ax=ax1)
ax1.set_axis_off()
# legend
from matplotlib.lines import Line2D
legend_els = ([Line2D([0],[0], marker='o', color='w',
                markerfacecolor=c, markersize=7, label=r)
                for r, c in ROLE_COLORS.items()] +
              [Line2D([0],[0], marker='o', color='w',
                markerfacecolor='#66BB6A', markersize=7, label='Plant')])
ax1.legend(handles=legend_els, loc="upper center",
           fontsize=7, facecolor="#0D1117", labelcolor=TXT,
           framealpha=0.8, ncol=3, bbox_to_anchor=(0.5,-0.02))
 

In [91]:
ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2, "Degree Distribution")
deg_plant = sorted([G_orig.degree(p) for p in plants], reverse=True)
deg_poll  = sorted([G_orig.degree(p) for p in pollinators], reverse=True)
ax2.plot(deg_plant, "o-", color="#66BB6A", markersize=4, lw=1.5, label="Plants")
ax2.plot(deg_poll,  "s-", color="#42A5F5", markersize=5, lw=1.5, label="Pollinators")
ax2.set_xlabel("Species rank", color=TXT)
ax2.set_ylabel("Degree", color=TXT)
ax2.legend(fontsize=8, facecolor="#0D1117", labelcolor=TXT)
ax2.grid(color=GRID, ls="--", lw=0.5)
 

In [92]:
ax3 = fig.add_subplot(gs[0, 2])
style_ax(ax3, "Species Roles (GNN-Lite Embedding)")
pca = PCA(n_components=2)
coords = pca.fit_transform(
    StandardScaler().fit_transform(
        metrics_df[["degree_centrality","betweenness_centrality",
                    "eigenvector_centrality","clustering_coefficient"]]))
for role, color in ROLE_COLORS.items():
    mask = metrics_df["role"] == role
    ax3.scatter(coords[mask, 0], coords[mask, 1],
                c=color, label=role, s=60, alpha=0.9, edgecolors="white", lw=0.4)
for i, sp in enumerate(metrics_df.index):
    ax3.annotate(sp[:8], (coords[i,0], coords[i,1]),
                 fontsize=4.5, color=TXT, alpha=0.7)
ax3.set_xlabel("PC1", color=TXT)
ax3.set_ylabel("PC2", color=TXT)
ax3.legend(fontsize=7, facecolor="#0D1117", labelcolor=TXT)
ax3.grid(color=GRID, ls="--", lw=0.5)

In [93]:
ax4 = fig.add_subplot(gs[1, 0])
style_ax(ax4, "Cascading Co-extinction")
ax4.plot(fr_deg, fs_deg, "r-o", markersize=3, lw=2,
         label="Targeted (high-degree)")
ax4.plot(fr_rnd, fs_rnd, "b-s", markersize=3, lw=2,
         label="Random removal")
ax4.axhline(0.5, color="yellow", ls="--", lw=1, alpha=0.7, label="50% threshold")
ax4.fill_between(fr_deg, fs_deg, alpha=0.15, color="red")
ax4.fill_between(fr_rnd, fs_rnd, alpha=0.10, color="blue")
ax4.set_xlabel("Fraction of pollinators removed")
ax4.set_ylabel("Fraction of plants surviving")
ax4.legend(fontsize=7.5, facecolor="#0D1117", labelcolor=TXT)
ax4.set_xlim(0, 1); ax4.set_ylim(0, 1.05)
ax4.grid(color=GRID, ls="--", lw=0.5)

In [94]:
ax5 = fig.add_subplot(gs[1, 1])
style_ax(ax5, "Network Fragmentation")
ax5.plot(fr_deg, nc_deg, "r-o", markersize=3, lw=2, label="Targeted")
ax5.plot(fr_rnd, nc_rnd, "b-s", markersize=3, lw=2, label="Random")
ax5.set_xlabel("Fraction of pollinators removed")
ax5.set_ylabel("Number of connected components")
ax5.legend(fontsize=7.5, facecolor="#0D1117", labelcolor=TXT)
ax5.grid(color=GRID, ls="--", lw=0.5)

In [95]:
ax6 = fig.add_subplot(gs[1, 2])
style_ax(ax6, "Climate Stress Simulation")
color1, color2 = "#FFA726", "#26C6DA"
l1, = ax6.plot(stress_lvls, ns_poll,  "-o", markersize=3, lw=2,
               color=color1, label="Pollinators")
l2, = ax6.plot(stress_lvls, ns_plant, "-s", markersize=3, lw=2,
               color=color2, label="Plants")
ax6_r = ax6.twinx()
ax6_r.plot(stress_lvls,
           [c * 100 for c in conn_list],
           "--", lw=1.5, color="#AB47BC", alpha=0.8, label="Connectance %")
ax6_r.set_ylabel("Connectance (%)", color="#AB47BC", fontsize=8)
ax6_r.tick_params(axis='y', colors="#AB47BC", labelsize=8)
ax6_r.spines['right'].set_edgecolor(GRID)
ax6.set_xlabel("Stress level (0=none, 1=total)")
ax6.set_ylabel("Species surviving")
lines = [l1, l2] + ax6_r.get_lines()
ax6.legend(lines, [l.get_label() for l in lines],
           fontsize=7.5, facecolor="#0D1117", labelcolor=TXT)
ax6.grid(color=GRID, ls="--", lw=0.5)

In [96]:
ax7 = fig.add_subplot(gs[2, 0])
style_ax(ax7, "Adaptive Rewiring vs. No Rewiring")
ax7.plot(steps_rw, surv_rw,  "-o", markersize=3, lw=2,
         color="#4CAF50", label=f"With rewiring (p=0.6)")
ax7.plot(steps_rw, surv_nrw, "-s", markersize=3, lw=2,
         color="#EF5350", label="No rewiring")
ax7.fill_between(steps_rw, surv_rw, surv_nrw,
                 alpha=0.2, color="#81C784",
                 label="Resilience gain")
ax7.set_xlabel("Fraction of pollinators removed")
ax7.set_ylabel("Fraction of plants surviving")
ax7.set_xlim(0, 1); ax7.set_ylim(0, 1.05)
ax7.legend(fontsize=7.5, facecolor="#0D1117", labelcolor=TXT)
ax7.grid(color=GRID, ls="--", lw=0.5)
 

In [97]:
ax8 = fig.add_subplot(gs[2, 1])
style_ax(ax8, "Early Warning: Variance")
ax8.plot(ew_stress, ew_var, "-o", markersize=3, lw=2,
         color="#FFCA28", label="Variance")
ax8.axvline(0.75, color="red", ls="--", lw=1.2, label="Critical zone")
ax8.fill_betweenx([min(ew_var), max(ew_var)],
                  0.75, 1.0, alpha=0.12, color="red")
ax8.set_xlabel("Environmental stress level")
ax8.set_ylabel("Variance in community biomass")
ax8.legend(fontsize=7.5, facecolor="#0D1117", labelcolor=TXT)
ax8.grid(color=GRID, ls="--", lw=0.5)

In [98]:
ax9 = fig.add_subplot(gs[2, 2])
style_ax(ax9, "Early Warning: Autocorrelation")
ax9.plot(ew_stress, ew_ac, "-s", markersize=3, lw=2,
         color="#AB47BC", label="Autocorrelation (lag-1)")
ax9.axhline(0.9, color="yellow", ls="--", lw=1, label="CSD threshold ~0.9")
ax9.axvline(0.75, color="red", ls="--", lw=1.2, alpha=0.6)
ax9.set_xlabel("Environmental stress level")
ax9.set_ylabel("Lag-1 autocorrelation")
ax9.set_ylim(-1.05, 1.05)
ax9.legend(fontsize=7.5, facecolor="#0D1117", labelcolor=TXT)
ax9.grid(color=GRID, ls="--", lw=0.5)
 
 

In [99]:
ax10 = fig.add_subplot(gs[3, 0])
style_ax(ax10, "Extinction Risk Ranking (ML)")
sorted_risk = risk_df["extinction_risk"].sort_values(ascending=False)
colors_bar  = [ROLE_COLORS.get(risk_df.loc[sp, "role"], "#888")
               for sp in sorted_risk.index]
bars = ax10.barh(range(len(sorted_risk)),
                 sorted_risk.values[::-1],
                 color=colors_bar[::-1], alpha=0.85)
ax10.set_yticks(range(len(sorted_risk)))
ax10.set_yticklabels([sp[:14] for sp in sorted_risk.index[::-1]],
                     fontsize=6.5, color=TXT)
ax10.set_xlabel("Extinction Risk Score")
ax10.set_xlim(0, 1.05)
ax10.grid(axis='x', color=GRID, ls="--", lw=0.5)
# role legend
legend_els2 = [Line2D([0],[0], marker='s', color='w',
                markerfacecolor=c, markersize=7, label=r)
               for r, c in ROLE_COLORS.items()]
ax10.legend(handles=legend_els2, loc="lower right",
            fontsize=6.5, facecolor="#0D1117", labelcolor=TXT)

In [100]:
ax11 = fig.add_subplot(gs[3, 1])
style_ax(ax11, "ML Feature Importance")
feat_imp_sorted = feat_imp.sort_values()
ax11.barh(feat_imp_sorted.index, feat_imp_sorted.values,
          color="#42A5F5", alpha=0.85)
ax11.set_xlabel("Importance")
ax11.grid(axis='x', color=GRID, ls="--", lw=0.5)

In [101]:
ax12 = fig.add_subplot(gs[3, 2])
style_ax(ax12, "Pollinator Community Similarity (Jaccard)")
im = ax12.imshow(sim_matrix.values, cmap="YlOrRd",
                 vmin=0, vmax=1, aspect="auto")
ticks = range(n_poll)
ax12.set_xticks(ticks)
ax12.set_yticks(ticks)
ax12.set_xticklabels([c[:8] for c in sim_matrix.columns],
                     rotation=90, fontsize=5.5, color=TXT)
ax12.set_yticklabels([c[:8] for c in sim_matrix.index],
                     fontsize=5.5, color=TXT)
cbar = plt.colorbar(im, ax=ax12, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=7, colors=TXT)
cbar.outline.set_edgecolor(GRID)

In [102]:
fig.suptitle(
    "Stability & Collapse of Pollination Networks — Full Analysis\n"
    f"Dataset M_PL_074_01  |  {n_plants} plants × {n_poll} hummingbird pollinators  "
    f"|  NODF={nestedness:.1f}  Q={Q:.3f}",
    color=TXT, fontsize=13, fontweight="bold", y=0.975
)
 
out_path = f"{OUT_DIR}/pollination_network_analysis.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"\n✓ Figure saved → {out_path}")


✓ Figure saved → ./outputs/pollination_network_analysis.png


In [103]:
summary = {
    "Network nodes": G_orig.number_of_nodes(),
    "Network edges": G_orig.number_of_edges(),
    "Plant species": n_plants,
    "Pollinator species": n_poll,
    "Nestedness (NODF)": round(nestedness, 2),
    "Modularity (Q)": round(Q, 4),
    "Communities": len(communities),
    "Connectance": round(G_orig.number_of_edges() / (n_plants * n_poll), 4),
    "Most connected pollinator": metrics_df["degree"].idxmax(),
    "Highest extinction risk": risk_df["extinction_risk"].idxmax(),
    "Stress at 50% connectivity loss": round(
        float(stress_lvls[next((i for i,c in enumerate(conn_list)
                               if c < conn_list[0]*0.5), -1)]), 3),
}
 
print("\n" + "═"*55)
print("SUMMARY")
print("═"*55)
for k, v in summary.items():
    print(f"  {k:<35} {v}")
print("═"*55)
 
# Save metrics CSV
full_features.to_csv(f"{OUT_DIR}/species_features_and_roles.csv")
risk_df[["degree","role","extinction_risk","rel_abundance"]]\
    .sort_values("extinction_risk", ascending=False)\
    .to_csv(f"{OUT_DIR}/extinction_risk_ranking.csv")
 
print(f"\n✓ CSV files saved to {OUT_DIR}")
print("Done ✓")
 


═══════════════════════════════════════════════════════
SUMMARY
═══════════════════════════════════════════════════════
  Network nodes                       185
  Network edges                       361
  Plant species                       84
  Pollinator species                  101
  Nestedness (NODF)                   18.66
  Modularity (Q)                      0.5067
  Communities                         11
  Connectance                         0.0426
  Most connected pollinator           Scaeva melanostoma
  Highest extinction risk             Andinus venustus
  Stress at 50% connectivity loss     0.931
═══════════════════════════════════════════════════════


NameError: name 'full_features' is not defined